In [0]:
from TornAPI.Torn import Faction
from pyspark.sql.functions import from_unixtime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, BooleanType, ArrayType

faction_api = Faction(dbutils.secrets.get("Personal", "TornAPI"))

In [0]:
if spark.catalog.tableExists("torn.faction.chains"):
    df = spark.read.table("torn.faction.chains")
    last_date = df.select("end").agg({"end": "max"}).collect()[0]["max(end)"]
    id_list = [id[0] for id in df.select("id").collect()]
else:
    last_date = 1681484237
    id_list = []

In [0]:
chain_data = faction_api.get_chains(ts_from=last_date)

schema = StructType([
    StructField("id", IntegerType()),
    StructField("chain", IntegerType()),
    StructField("respect", FloatType()),
    StructField("start", IntegerType()),
    StructField("end", IntegerType())
    ])

sp_chains_data = spark.createDataFrame(chain_data["chains"], schema = schema)

sp_chains_data = sp_chains_data.filter(~sp_chains_data["id"].isin(id_list))

sp_chains_data.write.format("delta").mode("append").saveAsTable("torn.faction.chains")